# FH-05 · Speech front-end (Phase 5): VAD, ASR, diarization, and speech → decision degradation
**GPU runtime.** Needs: `silero-vad`, `faster-whisper`, `pyannote.audio` (+ a Hugging Face token that has accepted the licence for `pyannote/speaker-diarization-3.1`), `gTTS`.

What is real and what is not - read before presenting:
- Audio here is **synthetic** (text-to-speech from the test conversations, two voices). That is an *easy* condition. It measures how much ASR errors alone hurt decisions and catches pipeline bugs.
- It is **not** a claim about real rooms, phone-quality audio, overlap or accents. For that, record 20-30 short scripted two-person conversations with consent (format described at the bottom) and rerun the same cells.
- Hinglish in Roman script is skipped for TTS (engines read it badly). English and Devanagari Hindi only.

In [ ]:
# ---- Colab setup: upload futurehuman_testing_suite.zip when asked (once per session) ----
import os, sys, subprocess
SUITE = "/content/futurehuman_testing_suite"
if not os.path.isdir(SUITE):
    from google.colab import files
    up = files.upload()
    subprocess.run(["unzip", "-o", "-q", list(up)[0], "-d", "/content"], check=True)
os.chdir(SUITE); sys.path.insert(0, SUITE)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "accelerate", "bitsandbytes", "tabulate"])
print("ready in", os.getcwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "faster-whisper", "silero-vad", "gTTS", "soundfile", "pyannote.audio", "pyannote.metrics"])
import json, numpy as np, pandas as pd, soundfile as sf, torch
pd.set_option("display.max_colwidth", 120)
from fh.dataset import load_actionability
from fh.speech import wer, cer, entity_error_rate, vad_metrics
DATA = "data/actionability_v1_adjudicated.jsonl" if os.path.exists("data/actionability_v1_adjudicated.jsonl") else "data/actionability_v1.jsonl"
cases = load_actionability(DATA)
pool = [c for c in cases if c["lang"] in ("en", "hindi") and len(c["turns"]) >= 2][:25] + [c for c in cases if c["lang"] == "hindi"][:6]
print(len(pool), "conversations for TTS")

## 1 · Make synthetic audio (two voices) + reference timings

In [ ]:
from gtts import gTTS
os.makedirs("audio", exist_ok=True)
def tts_turn(text, speaker, lang, path):
    tld = {"user": "co.in", "other": "co.uk"}[speaker] if lang == "en" else "co.in"
    gTTS(text, lang="en" if lang == "en" else "hi", tld=tld).save(path + ".mp3")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path + ".mp3", "-ar", "16000", "-ac", "1", path + ".wav"], check=True)
    return sf.read(path + ".wav")[0]
SR, items = 16000, []
for c in pool:
    lang = "hi" if c["lang"] == "hindi" else "en"; segs, wav, t = [], [], 0.0
    for i, (spk, text) in enumerate(c["turns"]):
        a = tts_turn(text, spk, lang, f"audio/{c['id']}_{i}"); d = len(a) / SR
        segs.append(dict(speaker=spk, start=t, end=t + d, text=text, wav=f"audio/{c['id']}_{i}.wav")); wav += [a, np.zeros(int(.5 * SR))]; t += d + .5
    sf.write(f"audio/{c['id']}_full.wav", np.concatenate(wav), SR)
    items.append(dict(id=c["id"], lang=lang, segs=segs, total=t, case=c))
print("audio built:", len(items))

## 2 · VAD (Silero): missed speech and false alarms

In [ ]:
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps
vad = load_silero_vad(); rows = []
for it in items:
    w = read_audio(f"audio/{it['id']}_full.wav", sampling_rate=SR)
    hyp = [(s["start"] / SR, s["end"] / SR) for s in get_speech_timestamps(w, vad, sampling_rate=SR)]
    rows.append(vad_metrics([(s["start"], s["end"]) for s in it["segs"]], hyp, it["total"]))
v = pd.DataFrame(rows); print(v.mean().round(3)); print("worst clip speech recall:", v.speech_recall.min().round(3))

## 3 · ASR (faster-whisper large-v3): WER, CER, entity errors, hallucination on silence

In [ ]:
from faster_whisper import WhisperModel
asr = WhisperModel("large-v3", device="cuda", compute_type="float16")
def transcribe(path, lang):
    segs, _ = asr.transcribe(path, language=lang, beam_size=5, vad_filter=False); return " ".join(s.text.strip() for s in segs)
rows = []
for it in items:
    for s in it["segs"]:
        hyp = transcribe(s["wav"], it["lang"]); s["asr"] = hyp
        ents = [str(v) for v in (it["case"].get("entities") or {}).values()] if s is it["segs"][-1] else []
        rows.append(dict(id=it["id"], lang=it["lang"], ref=s["text"], hyp=hyp, wer=wer(s["text"], hyp), cer=cer(s["text"], hyp), ent_err=entity_error_rate(ents, hyp)))
a = pd.DataFrame(rows); print(a.groupby("lang")[["wer", "cer", "ent_err"]].mean().round(3))
sil = np.zeros(SR * 5, dtype="float32"); sf.write("audio/silence.wav", sil, SR); print("hallucination on 5 s of silence ->", repr(transcribe("audio/silence.wav", "en")))
a.sort_values("wer", ascending=False).head(8)[["lang", "ref", "hyp", "wer"]]

## 4 · Speech → decision degradation (the number that matters)
Same conversations, but every turn replaced by its ASR transcript. Compare with the clean-text decision from FH-01.

In [ ]:
import copy
TAG = "base"; ADAPTER = None; QUANT = "4bit"
from fh.model import load_qwen, make_gen
from fh.evaluate import run_cases
from fh.prompts import select_fewshot
from fh.metrics import safety
model, tok = load_qwen("Qwen/Qwen3-4B", quant=QUANT, adapter=ADAPTER); gen = make_gen(model, tok)
dev = [c for c in cases if c["split"] == "dev"]; fs = select_fewshot(dev, n=(3, 3, 2), seed=0)
clean_cases = [it["case"] for it in items]
asr_cases = []
for it in items:
    c2 = copy.deepcopy(it["case"]); c2["turns"] = [[s["speaker"], s["asr"]] for s in it["segs"]]; asr_cases.append(c2)
d_clean = run_cases(clean_cases, gen, "A1_improved", fewshot=fs, progress=False); d_asr = run_cases(asr_cases, gen, "A1_improved", fewshot=fs, progress=False)
m = d_clean.merge(d_asr, on="id", suffixes=("_clean", "_asr"))
print("decision flips caused by ASR errors:", int((m.pred_label_clean != m.pred_label_asr).sum()), "of", len(m))
for nm, d in (("clean text", d_clean), ("ASR text", d_asr)):
    s_ = safety(d.gold_label.tolist(), d.pred_label.tolist()); print(f"{nm:11s} accuracy {s_['accuracy']['rate']:.3f} | false-action {s_['false_action']['k']}/{s_['false_action']['n']} | missed {s_['missed_action']['k']}/{s_['missed_action']['n']}")
m[m.pred_label_clean != m.pred_label_asr][["id", "gold_label_clean", "pred_label_clean", "pred_label_asr"]]

## 5 · Diarization (pyannote 3.1): DER on synthetic two-voice audio (easy case only)

In [ ]:
from getpass import getpass
from pyannote.audio import Pipeline
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate
HF = getpass("Hugging Face token (accepted pyannote/speaker-diarization-3.1 terms): ")
try: pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF)
except TypeError: pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=HF)
pipe.to(torch.device("cuda")); der = DiarizationErrorRate(); rows = []
for it in [i for i in items if i["lang"] == "en" and len(i["segs"]) >= 3][:15]:
    ref = Annotation()
    for s in it["segs"]: ref[Segment(s["start"], s["end"])] = s["speaker"]
    hyp = pipe(f"audio/{it['id']}_full.wav"); hyp = getattr(hyp, "speaker_diarization", hyp)
    rows.append(dict(id=it["id"], der=der(ref, hyp)))
dd = pd.DataFrame(rows); print("mean DER (synthetic, easy):", dd.der.mean().round(3), "| worst:", dd.der.max().round(3))

## Using REAL recordings (recommended before any claim about the product)
Put files in `real_audio/` and a `real_audio/index.csv` with columns `file, lang, speaker_turns_json, reference_text`, where `speaker_turns_json` is
`[["user", 0.0, 2.8], ["other", 3.1, 6.0]]`. Record on the target laptop, with real background noise, 2-3 people, Indian-accented English/Hindi/Hinglish, and some overlap.
Then reuse sections 2, 3 and 5 unchanged (read `real_audio/*.wav` instead of `audio/*_full.wav`).